In [ ]:
# convert the transforms .npy files into a csv file for easier analysis and tracking of the dataset.

import os
import numpy as np
import glob
import csv
from tqdm import tqdm

transforms_dir = "./sample_dataset/transforms"
output_csv = "./sample_dataset/metadata.csv"

distances = [5, 7, 10]
pitches = [10, 30, 45]
yaws = list(range(0, 360, 30))

def get_sequence_index(dist, pitch, yaw):
    # map camera params for sequence tracking.
    try:
        d_idx = distances.index(int(dist))
        p_idx = pitches.index(int(pitch))
        y_idx = yaws.index(int(yaw))
    except ValueError:
        return -1 
    
    return d_idx * (len(pitches) * len(yaws)) + p_idx * len(yaws) + y_idx

def main():
    npy_files = sorted(glob.glob(os.path.join(transforms_dir, "*.npy")))
    print(f"Found {len(npy_files)} transform files to process.")

    current_vehicle = None
    prev_seq_index = -1
    sampleset_no = 0 
    location_id = 0
    
    # data header row
    csv_data = [["filename", "distance", "pitch", "yaw", "vehicle_id", "sampleset_no", "location_id"]]

    for file in tqdm(npy_files, desc="Extracting Metadata"):
        try:
            # read original file
            data = np.load(file)
            dist, pitch, yaw, vehicle = data[0], data[1], data[2], data[3]
            
            # extract filename
            base_filename = os.path.splitext(os.path.basename(file))[0]
            
            seq_index = get_sequence_index(dist, pitch, yaw)
            
            if vehicle != current_vehicle:
                current_vehicle = vehicle
                sampleset_no += 1
                location_id = 1
            elif seq_index <= prev_seq_index:
                sampleset_no += 1
                location_id += 1
            
            prev_seq_index = seq_index
            
            # append to list
            csv_data.append([base_filename, dist, pitch, yaw, vehicle, sampleset_no, location_id])
            
        except Exception as e:
            print(f"Error reading {file}: {e}")

    # write everything to csv
    print(f"\nWriting to {output_csv}...")
    with open(output_csv, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerows(csv_data)

    print("Metadata extraction done")

if __name__ == '__main__':
    main()